# 004 part2：cosine 档 → R4 重排 → 解冻续训 → merged/forward 对照（T4，一次跑完）

Drive 文件夹放好 9 个文件：`reference-fpnet-v32.py`、`np_queries_1184.json`、
`generic_queries.json`、`train_structures.jsonl`、`train_shard_200k_p0..p3.parquet`、
`head_rank_5000.pt`。按顺序逐格跑，总计约 40 分钟。产物写 `/content/004-output2/`。

范围说明：de novo 生成分支与 Kaggle 离线提交**不在本 notebook**（前者需分子式+生成器
权重决策，后者是 Kaggle 平台）。forward 只做有界试点，无已核实权重 URL 时按许可规则跳过并记录。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DATA = Path('/content/drive/MyDrive/Colab Notebooks/enveda')
OUT = Path('/content/004-output2'); OUT.mkdir(parents=True, exist_ok=True)
need = ['reference-fpnet-v32.py','np_queries_1184.json','generic_queries.json',
        'train_structures.jsonl','head_rank_5000.pt'] + ['train_shard_200k_p%d.parquet'%i for i in range(4)]
missing = [f for f in need if not (DATA/f).exists()]
print('missing:', missing)
assert not missing, missing
print('all inputs present')


In [ ]:
import importlib.util, subprocess, sys
missing = [n for n in ('torch','numpy','pyarrow','kagglehub','rdkit')
           if importlib.util.find_spec(n) is None]
if missing:
    subprocess.run([sys.executable,'-m','pip','install',*missing], check=True)
print({'installed_missing': missing})
import platform, torch
print('python =', sys.version.split()[0])
print('torch =', torch.__version__, '| cuda =', torch.cuda.is_available())
assert torch.cuda.is_available(), 'need T4 GPU runtime'
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory/2**30,3), 'GiB')
import pyarrow as pa, rdkit
print('pyarrow =', pa.__version__, '| rdkit =', rdkit.__version__)


In [ ]:
import kagglehub, time
from pathlib import Path
import os
roots = {}
for h in ['prvsiyan/casmi26-fp-models-v2/versions/1',
          'prvsiyan/coconut-casmi26-candidates/versions/2']:
    t0=time.perf_counter(); p=Path(kagglehub.dataset_download(h))
    roots[h]=p
    print(h, '->', p, round(time.perf_counter()-t0,1))
CACHE_FPV2 = roots['prvsiyan/casmi26-fp-models-v2/versions/1']
CACHE_CAND = roots['prvsiyan/coconut-casmi26-candidates/versions/2']
for f,exp in [('fp_single_s2.pt',144110255),('fp_merged_m1.pt',144110255)]:
    p = CACHE_FPV2/f
    print(f, p.stat().st_size, 'OK' if p.stat().st_size==exp else 'MISMATCH')
    assert p.stat().st_size==exp
b = CACHE_CAND/'fp_bits.npy'
print('fp_bits', b.stat().st_size); assert b.stat().st_size==55568


In [ ]:
import importlib.util, json, torch
import numpy as np
spec = importlib.util.spec_from_file_location('refv32', str(DATA/'reference-fpnet-v32.py'))
ref = importlib.util.module_from_spec(spec); spec.loader.exec_module(ref)
def load_net(fname, head_pt=None):
    ck = torch.load(CACHE_FPV2/fname, map_location='cpu', weights_only=True)
    n = ref.FPNet(ck['nbits'], d=ck['d'], layers=ck['layers']).cuda().eval()
    n.load_state_dict(ck['model'], strict=True)
    if head_pt:
        h = torch.load(head_pt, map_location='cpu')
        hd = h.get('head', h)
        n.load_state_dict({('head.'+k if not k.startswith('head.') else k): v for k,v in hd.items()}, strict=False)
    return n, ck
net_pub, ck0 = load_net('fp_single_s2.pt')
net_rank, _ = load_net('fp_single_s2.pt', str(DATA/'head_rank_5000.pt'))
net_merged, ckm = load_net('fp_merged_m1.pt')
print('nets ok:', ck0['nbits'], ck0['d'], ck0['layers'])
BITS = np.load(CACHE_CAND/'fp_bits.npy')
print('BITS:', BITS.shape)


In [ ]:
import json
import numpy as np
from multiprocessing import Pool as MPool
from rdkit import RDLogger
from rdkit.Chem import rdFingerprintGenerator, MACCSkeys
from rdkit.Chem.Descriptors import ExactMolWt
RDLogger.DisableLog('rdApp.*')
recs = [json.loads(l) for l in open(str(DATA/'train_structures.jsonl'))]
print('structures:', len(recs))
assert len(recs) == 275810
def _init():
    global _m2,_m3,_rk
    _m2 = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=4096)
    _m3 = rdFingerprintGenerator.GetMorganGenerator(radius=3, fpSize=4096)
    _rk = rdFingerprintGenerator.GetRDKitFPGenerator(fpSize=2048, maxPath=6)
def one(r):
    import numpy as _np
    from rdkit import Chem
    m = Chem.MolFromSmiles(r['smiles'])
    if m is None: return None
    try:
        fp = _np.concatenate([_m2.GetFingerprintAsNumPy(m).astype(_np.uint8),
                              _m3.GetFingerprintAsNumPy(m).astype(_np.uint8),
                              _rk.GetFingerprintAsNumPy(m).astype(_np.uint8),
                              _np.array(MACCSkeys.GenMACCSKeys(m), dtype=_np.uint8)])[BITS].astype(_np.float32)
        return (r['inchikey14'], fp, float(ExactMolWt(m)))
    except Exception:
        return None
Fp,Mm,Kk,fail = [],[],[],0
with MPool(4, initializer=_init) as p:
    for i in range(0, len(recs), 25000):
        out = p.map(one, recs[i:i+25000], chunksize=500)
        Fp.append(np.packbits(np.array([x[1] if x else np.zeros(6930,np.uint8) for x in out], dtype=np.uint8), axis=1))
        Mm.append(np.array([x[2] if x else np.nan for x in out], dtype=np.float64))
        Kk += [x[0] if x else '' for x in out]
        fail += sum(1 for x in out if x is None)
        print('pool %d/%d fail=%d' % (min(i+25000,len(recs)), len(recs), fail))
pool_fp = np.vstack(Fp); pool_mass = np.concatenate(Mm); pool_keys = np.array(Kk)
np.save('/content/pool_fp.npy', pool_fp); np.save('/content/pool_mass.npy', pool_mass); np.save('/content/pool_keys.npy', pool_keys)
key_to_idx = {k:i for i,k in enumerate(pool_keys) if k}
print(pool_fp.shape, pool_mass.shape, 'indexed:', len(key_to_idx))
Q = json.load(open(str(DATA/'np_queries_1184.json')))
GQ = json.load(open(str(DATA/'generic_queries.json')))['spectra']
print('Q:', len(Q), 'GQ:', len(GQ))


## 共用打分件（mass 窗 + 单谱前向 + cosine，供后文复用）

In [ ]:
MASS = dict(C=12.0,H=1.00782503207,N=14.0030740048,O=15.9949146196,P=30.97376163,
    S=31.97207100,F=18.99840322,Cl=34.96885268,Br=78.9183371,I=126.904473,
    Na=22.9897692809,K=38.96370668,Si=27.9769265325,B=11.0093054,Se=79.9165213)
E=0.00054857990; PROTON=MASS['H']-E; H2O=2*MASS['H']+MASS['O']
NH4=MASS['N']+4*MASS['H']; FORMATE=MASS['C']+2*MASS['H']+2*MASS['O']
ACETATE=2*MASS['C']+4*MASS['H']+2*MASS['O']
ADDUCTS = {'[M+H]+':(1,1,PROTON),'[M+NH4]+':(1,1,NH4-E),"[M+Na]+":(1,1,MASS['Na']-E),
 '[M+K]+':(1,1,MASS['K']-E),'[M-H2O+H]+':(1,1,PROTON-H2O),'[M-2H2O+H]+':(1,1,PROTON-2*H2O),
 '[M+2H]2+':(1,2,2*PROTON),'[M]+':(1,1,-E),'[M-H2O]+':(1,1,-E-H2O),
 '[M+CH3OH+H]+':(1,1,PROTON+MASS['C']+4*MASS['H']+MASS['O']),
 '[M+CH3CN+H]+':(1,1,PROTON+2*MASS['C']+3*MASS['H']+MASS['N']),
 '[M-H]-':(1,1,-PROTON),'[M-H2O-H]-':(1,1,-PROTON-H2O),'[M+CH2O2-H]-':(1,1,FORMATE-PROTON),
 '[M+C2H4O2-H]-':(1,1,ACETATE-PROTON),"[M+Cl]-":(1,1,MASS['Cl']+E),'[M]-':(1,1,E),
 '[M-2H]-':(1,2,-2*PROTON),'[M+Na-2H]-':(1,1,MASS['Na']-2*PROTON),
 '[2M+H]+':(2,1,PROTON),'[2M+Na]+':(2,1,MASS['Na']-E),'[2M+NH4]+':(2,1,NH4-E),
 '[2M+K]+':(2,1,MASS['K']-E),'[2M-H]-':(2,1,-PROTON),'[2M+CH2O2-H]-':(2,1,FORMATE-PROTON),
 '[2M+C2H4O2-H]-':(2,1,ACETATE-PROTON),'[2M+Na-2H]-':(2,1,MASS['Na']-2*PROTON),
 '[3M+H]+':(3,1,PROTON),'[3M-H]-':(3,1,-PROTON)}
def neutral_mass(mz, adduct):
    out = np.full(len(mz), np.nan); ad = np.asarray(adduct, dtype=object)
    for a,(n,z,d) in ADDUCTS.items():
        m = (ad == a)
        if m.any(): out[m] = (mz[m]*z - d)/n
    return out
def window(qm, ppm=20.0):
    return np.where(np.abs(pool_mass - qm) <= qm*ppm*1e-6)[0]
def prep_row(row):
    a,b = ref.prep_peaks(np.asarray(row['ms2_mzs'],float), np.asarray(row['ms2_normalized_intensities'],float), float(row['precursor_mz']))
    return a,b
def fwd1(model, row):
    import numpy as _np, torch as _t
    a,b = prep_row(row)
    if not len(a): return None
    N = len(a)
    mz=_np.zeros((1,N),_np.float32); it=_np.zeros((1,N),_np.float32); pad=_np.zeros((1,N),bool)
    mz[0,:N]=a; it[0,:N]=b
    ce = row.get('collision_energy_ev')
    ce = float(_np.mean(_np.atleast_1d(ce))) if ce else 25.0
    T = lambda x: _t.as_tensor(x, device='cuda')
    args=(T(mz),T(it),T(pad),T(_np.array([float(row['precursor_mz'])],_np.float32)),
          T(_np.array([ref.ADDUCT_IX.get(row['adduct'],ref.ADDUCT_IX['<unk>'])])),
          T(_np.array([ref.instr_family(row['instrument_type'])])),
          T(_np.array([ce],_np.float32)),
          T(_np.array([1.0 if row['ionization_mode']=='positive' else -1.0],_np.float32)))
    model.eval()
    with _t.no_grad():
        return model(*args).float().cpu().numpy()[0]
def cos_score(qmz, qit, lmz, lit, ppm=20.0):
    import numpy as _np
    qi = _np.argsort(-qit)[:128]; qm, qi_ = qmz[qi], qit[qi]
    li = _np.argsort(-lit)[:128]; lm, li_ = lmz[li], lit[li]
    qn = _np.sqrt(qi_/qi_.max()); ln = _np.sqrt(li_/li_.max())
    qn = qn/_np.linalg.norm(qn); ln = ln/_np.linalg.norm(ln)
    o = _np.argsort(qm); qm, qn = qm[o], qn[o]
    o = _np.argsort(lm); lm, ln = lm[o], ln[o]
    i = j = 0; s = 0.0
    while i < len(qm) and j < len(lm):
        tol = max(qm[i], lm[j])*ppm*1e-6
        if abs(qm[i]-lm[j]) <= tol:
            s += qn[i]*ln[j]; i += 1; j += 1
        elif qm[i] < lm[j]: i += 1
        else: j += 1
    return float(s)
print('helpers ok')


## R1-cosine 档： shard 谱库余弦（20ppm 窗内，约 5 分钟）

In [ ]:
import json, time
import numpy as np
import pyarrow.parquet as pq
import pyarrow as pa
files = [str(DATA/'train_shard_200k_p%d.parquet')%i for i in range(4)]
import pyarrow as pa
LT = pa.concat_tables([pq.read_table(f) for f in files]).to_pydict()
LN = len(LT['inchikey14'])
print('library spectra:', LN)
Lmass = neutral_mass(np.array(LT['precursor_mz'], float), LT['adduct'])
res_c = []
t0 = time.perf_counter()
for qi, row in enumerate(Q):
    qm = neutral_mass(np.array([row['precursor_mz']]), [row['adduct']])[0]
    cand = np.where(np.abs(Lmass - qm) <= qm*20e-6)[0]
    qmz = np.asarray(row['ms2_mzs'], float); qit = np.asarray(row['ms2_normalized_intensities'], float)
    scored = []
    for j in cand:
        try:
            s = cos_score(qmz, qit, np.asarray(LT['ms2_mzs'][j], float), np.asarray(LT['ms2_normalized_intensities'][j], float))
        except Exception:
            continue
        scored.append((s, LT['inchikey14'][j]))
    scored.sort(reverse=True)
    top = scored[:25]
    rk = next((i+1 for i,(_,k) in enumerate(top) if k == row['inchikey14']), None)
    res_c.append({'id': qi, 'key': row['inchikey14'], 'win': int(len(cand)), 'rank': rk})
    if (qi+1) % 200 == 0: print(qi+1, round(time.perf_counter()-t0, 1))
json.dump(res_c, open('/content/004-output2/r1_cosine_np1184.json', 'w'))
rk = [r['rank'] for r in res_c if r['rank']]
print('cosine n=%d ranked=%d mrr=%.4f top1=%.4f' % (len(res_c), len(rk), sum(1/x for x in rk)/len(res_c), sum(1 for x in rk if x==1)/len(res_c)))


## R4：简单重排器（GEN 拟合 → NP 评估，约 3 分钟）
特征 `[fp_dot, cosine_vs_best_lib, neg_abs_ppm, adduct_match, instr_match]`；numpy 二分类 logistic 回归；样本来自 GEN（独立组），评估 NP。

In [ ]:
import json
import numpy as np
def feats_for(row, cand_keys, cand_fp, cand_meta):
    z = fwd1(net_rank, row)
    if z is None:
        return []
    qm = neutral_mass(np.array([row['precursor_mz']]), [row['adduct']])[0]
    F = cand_fp.astype(np.float32)
    dots = F @ z
    out = []
    for j, k in enumerate(cand_keys):
        cm = cand_meta[j]
        out.append([float(dots[j]), float(cm['cos']), abs(qm-cm['mass'])/qm*1e6,
                    1.0 if cm['adduct']==row['adduct'] else 0.0,
                    1.0 if cm['instr']==row['instrument_type'] else 0.0, k])
    return out
key_rows_LT = {}
for _j, _k in enumerate(LT['inchikey14']):
    key_rows_LT.setdefault(_k, []).append(_j)
print('shard keys:', len(key_rows_LT))
def window_members(row):
    qm = neutral_mass(np.array([row['precursor_mz']]), [row['adduct']])[0]
    w = window(qm)
    qmz = np.asarray(row['ms2_mzs'], float); qit = np.asarray(row['ms2_normalized_intensities'], float)
    meta = []
    for j in w:
        best, ba, bi, nl = 0.0, '?', '?', 0
        Lrows = key_rows_LT.get(pool_keys[j], [])
        nl = len(Lrows)
        for r in Lrows[:8]:
            try:
                s = cos_score(qmz, qit, np.asarray(LT['ms2_mzs'][r], float), np.asarray(LT['ms2_normalized_intensities'][r], float))
            except Exception:
                continue
            if s > best:
                best = s; ba = LT['adduct'][r]; bi = LT['instrument_type'][r]
        meta.append({'cos': best, 'mass': float(pool_mass[j]), 'adduct': ba, 'instr': bi, 'nlib': nl})
    return w, meta
X, y = [], []
for row in GQ:
    qm = neutral_mass(np.array([row['precursor_mz']]), [row['adduct']])[0]
    w = window(qm)
    if not len(w): continue
    F = np.unpackbits(pool_fp[w], axis=1)[:, :6930].astype(np.float32)
    _, meta = window_members(row)
    for f, m, jj in zip(feats_for(row, [pool_keys[j] for j in w], F, meta), meta, w):
        X.append(f[:5]); y.append(1.0 if pool_keys[jj] == row['inchikey14'] else 0.0)
X = np.array(X, float); y = np.array(y, float)
print('ranker rows:', X.shape, 'pos rate:', round(y.mean(), 4))
mu, sd = X.mean(0), X.std(0) + 1e-9
Xs = (X - mu)/sd
wgt = np.zeros(6)
lr = 0.1
for it in range(300):
    p = 1/(1+np.exp(-(Xs@wgt[:5]+wgt[5])))
    g = np.concatenate([(Xs*(p-y)[:,None]).mean(0), [(p-y).mean()]])
    wgt -= lr*g
ranker = {'w': wgt[:5].tolist(), 'b': float(wgt[5]), 'mu': mu.tolist(), 'sd': sd.tolist()}
json.dump(ranker, open('/content/004-output2/r4_ranker.json', 'w'))
rr = []
for row in Q:
    qm = neutral_mass(np.array([row['precursor_mz']]), [row['adduct']])[0]
    w = window(qm)
    if not len(w):
        rr.append(None); continue
    F = np.unpackbits(pool_fp[w], axis=1)[:, :6930].astype(np.float32)
    _, meta = window_members(row)
    Ff = feats_for(row, [pool_keys[j] for j in w], F, meta)
    Xn = [(np.array(f[:5])-mu)/sd for f in Ff]
    sc = [float(np.dot(x, ranker['w'])+ranker['b']) for x in Xn]
    order = np.argsort([-s for s in sc])
    ti = row['inchikey14']
    rk = next((int(i)+1 for i in order if pool_keys[w[i]] == ti), None)
    rr.append(rk)
got = [x for x in rr if x]
print('R4 NP mrr=%.4f top1=%.4f ranked=%d/%d' % (sum(1/x for x in got)/len(rr), sum(1 for x in got if x==1)/len(rr), len(got), len(rr)))
json.dump({'ranker': ranker, 'ranks': rr}, open('/content/004-output2/r4_np1184.json', 'w'))


## 解冻末两层续训（2000 步，约 5 分钟）：从 rank head 起，lr 1e-4，BCE+rank

In [ ]:
import json, time, random
import numpy as np, torch
from multiprocessing import Pool as MPool
torch.manual_seed(13); random.seed(13); np.random.seed(13)
ck = torch.load(CACHE_FPV2/'fp_single_s2.pt', map_location='cpu', weights_only=True)
net4 = ref.FPNet(ck['nbits'], d=ck['d'], layers=ck['layers']).cuda().train()
net4.load_state_dict(ck['model'], strict=True)
h = torch.load(str(DATA/'head_rank_5000.pt'), map_location='cpu')
net4.load_state_dict({('head.'+k if not k.startswith('head.') else k): v for k,v in h.get('head',h).items()}, strict=False)
keep = [n for n,_ in net4.named_parameters() if n.startswith('head') or 'blocks.4' in n or 'blocks.5' in n or n.startswith('norm')]
for n,p_ in net4.named_parameters():
    p_.requires_grad = n in keep
print('trainable:', sum(p.numel() for p in net4.parameters() if p.requires_grad))
smi_of = {}
for k,s in zip(LT['inchikey14'], LT['normalized_smiles']):
    if k not in smi_of: smi_of[k] = s
print('distinct:', len(smi_of))
def _init():
    global _m2,_m3,_rk
    _m2 = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=4096)
    _m3 = rdFingerprintGenerator.GetMorganGenerator(radius=3, fpSize=4096)
    _rk = rdFingerprintGenerator.GetRDKitFPGenerator(fpSize=2048, maxPath=6)
def one(kv):
    import numpy as _np
    from rdkit import Chem
    k,s = kv
    m = Chem.MolFromSmiles(s)
    if m is None: return (k,None)
    try:
        fp = _np.concatenate([_m2.GetFingerprintAsNumPy(m).astype(_np.uint8),
                              _m3.GetFingerprintAsNumPy(m).astype(_np.uint8),
                              _rk.GetFingerprintAsNumPy(m).astype(_np.uint8),
                              _np.array(MACCSkeys.GenMACCSKeys(m), dtype=_np.uint8)])[BITS].astype(_np.float32)
        return (k,fp)
    except Exception:
        return (k,None)
with MPool(4, initializer=_init) as p:
    items = list(smi_of.items())
    res_t = []
    for i in range(0, len(items), 20000):
        res_t += p.map(one, items[i:i+20000], chunksize=500)
        print('tgt %d/%d' % (min(i+20000,len(items)), len(items)))
tgt4 = {k:v for k,v in res_t if v is not None}
print('targets:', len(tgt4))
opt4 = torch.optim.AdamW([pp for pp in net4.parameters() if pp.requires_grad], lr=1e-4, weight_decay=0.01)
sc4 = torch.amp.GradScaler('cuda')
bce4 = torch.nn.BCEWithLogitsLoss()
ce4 = torch.nn.CrossEntropyLoss()
SQ4 = float(np.sqrt(6930.0))
def ce_of4(v):
    try: return float(np.mean(np.atleast_1d(v))) if v is not None and len(np.atleast_1d(v)) else 25.0
    except Exception: return 25.0
def step4(bs=32):
    idx = random.sample(range(len(LT['inchikey14'])), bs)
    P,Tg = [],[]
    for i in idx:
        a,b = ref.prep_peaks(np.asarray(LT['ms2_mzs'][i],float), np.asarray(LT['ms2_normalized_intensities'][i],float), float(LT['precursor_mz'][i]))
        if not len(a) or LT['inchikey14'][i] not in tgt4: continue
        P.append((a,b,i)); Tg.append(tgt4[LT['inchikey14'][i]])
    B = len(P); Np = max(len(a) for a,_,_ in P)
    mz=np.zeros((B,Np),np.float32); it=np.zeros((B,Np),np.float32); pad=np.ones((B,Np),bool)
    for j,(a,b,_) in enumerate(P):
        mz[j,:len(a)]=a; it[j,:len(b)]=b; pad[j,:len(a)]=False
    T = lambda x: torch.as_tensor(x, device='cuda')
    ii=[i for _,_,i in P]
    args=(T(mz),T(it),T(pad),T(np.array([float(LT['precursor_mz'][i]) for i in ii],np.float32)),
          T(np.array([ref.ADDUCT_IX.get(LT['adduct'][i],ref.ADDUCT_IX['<unk>']) for i in ii])),
          T(np.array([ref.instr_family(LT['instrument_type'][i]) for i in ii])),
          T(np.array([ce_of4(LT['collision_energy_ev'][i]) for i in ii],np.float32)),
          T(np.array([1.0 if LT['ionization_mode'][i]=='positive' else -1.0 for i in ii],np.float32)))
    y = T(np.stack(Tg))
    opt4.zero_grad(set_to_none=True)
    with torch.amp.autocast('cuda', dtype=torch.float16):
        out = net4(*args)
        lbce = bce4(out, y)
    zf = out.float()
    rl = []
    for j in range(len(P)):
        rkey = LT['inchikey14'][P[j][2]]
        qm = neutral_mass(np.array([LT['precursor_mz'][P[j][2]]]), [LT['adduct'][P[j][2]]])[0]
        w = np.where(np.abs(pool_mass - qm) <= qm*10e-6)[0]
        w = w[pool_keys[w] != rkey][:5000]
        if len(w) == 0: continue
        take = np.random.choice(w, min(63, len(w)), replace=False)
        F = np.unpackbits(pool_fp[np.concatenate([[key_to_idx[rkey]], take])], axis=1)[:, :6930].astype(np.float32)
        s = (torch.as_tensor(F, device='cuda') @ zf[j])
        s = (s - s.mean()) / SQ4
        rl.append(ce4(s.unsqueeze(0), torch.zeros(1, dtype=torch.long, device='cuda')))
    lrank = torch.stack(rl).mean() if rl else torch.zeros((), device='cuda')
    loss = lbce + 1.0 * lrank
    sc4.scale(loss).backward()
    sc4.step(opt4); sc4.update()
    return float(loss.detach()), float(lrank.detach()) if rl else 0.0
@torch.no_grad()
def eval42(model, Qs):
    model.eval(); rr = []
    for row in Qs:
        qm = neutral_mass(np.array([row['precursor_mz']]), [row['adduct']])[0]
        w = window(qm)
        if not len(w):
            rr.append(None); continue
        z = fwd1(model, row)
        if z is None:
            rr.append(None); continue
        F = np.unpackbits(pool_fp[w], axis=1)[:, :6930].astype(np.float32)
        order = np.argsort(-(F @ z))
        ti = key_to_idx.get(row['inchikey14'])
        pos = np.where(w[order]==ti)[0]
        rr.append(int(pos[0])+1 if len(pos) else None)
    model.train()
    got=[x for x in rr if x]
    return sum(1/x for x in got)/len(rr), sum(1 for x in got if x==1)/len(rr)
log4 = {'seed':13,'mode':'unfreeze_last2_bce_rank','bs':32,'lr':1e-4,'K':63,'steps':2000,'eval_every':500,'history':[]}
for tag,Qs in [('NP',Q),('GEN',GQ)]:
    m,v = eval42(net4, Qs)
    log4['history'].append({'step':0,'split':tag,'mrr':m,'top1':v})
    print('step 0 %s mrr=%.4f top1=%.4f' % (tag,m,v))
t0=time.perf_counter()
for s in range(1,2001):
    loss,lr_ = step4(32)
    if s%500==0:
        h = {'step':s,'loss':loss,'rank':lr_,'elapsed':round(time.perf_counter()-t0,1)}
        for tag,Qs in [('NP',Q),('GEN',GQ)]:
            m,v = eval42(net4, Qs)
            h[tag] = {'mrr':m,'top1':v}
        log4['history'].append(h)
        print('step %d loss=%.4f rank=%.4f NP=%.4f/%.4f GEN=%.4f/%.4f elapsed=%.0fs' % (s,loss,lr_,h['NP']['mrr'],h['NP']['top1'],h['GEN']['mrr'],h['GEN']['top1'],time.perf_counter()-t0))
torch.save({'tunable':{n:p.cpu() for n,p in net4.named_parameters() if p.requires_grad},'config':log4}, '/content/004-output2/unfreeze_2000.pt')
json.dump(log4, open('/content/004-output2/unfreeze_log.json','w'), indent=1)
print('saved unfreeze_2000.pt + unfreeze_log.json')


## merged 对照：同分子合并峰表走 merged 模型（约 2 分钟）
合并规则 mirror v32 `_merge_peaks`（近重复 m/z<0.005 留强峰）；中位前体、首谱 adduct/instrument、ce 25、平均 mode。只判 shape/finite 与排名，不比模型优劣。

In [ ]:
import json
import numpy as np, torch
from collections import defaultdict
G = defaultdict(list)
for i,row in enumerate(Q):
    G[row['inchikey14']].append(row)
print('molecules:', len(G))
def merge_peaks(rows):
    mz = np.concatenate([np.asarray(r['ms2_mzs'],float) for r in rows])
    it = np.concatenate([np.asarray(r['ms2_normalized_intensities'],float)/max(float(np.asarray(r['ms2_normalized_intensities'],float).max()),1e-9) for r in rows])
    o = np.argsort(mz); mz,it = mz[o],it[o]
    keep = np.ones(len(mz), bool)
    for j in range(1, len(mz)):
        if mz[j]-mz[j-1] < 0.005:
            if it[j] >= it[j-1]: keep[j-1] = False
            else: keep[j] = False
    return mz[keep], it[keep]
@torch.no_grad()
def merged_z(rows):
    import numpy as _np, torch as _t
    mz,it = merge_peaks(rows)
    a,b = ref.prep_peaks(mz, it, float(np.median([r['precursor_mz'] for r in rows])))
    if not len(a): return None
    N = len(a)
    m0 = rows[0]
    M=_np.zeros((1,N),_np.float32); I=_np.zeros((1,N),_np.float32); pad=_np.zeros((1,N),bool)
    M[0,:N]=a; I[0,:N]=b
    mode = float(_np.mean([1.0 if r['ionization_mode']=='positive' else -1.0 for r in rows]))
    T = lambda x: _t.as_tensor(x, device='cuda')
    args=(T(M),T(I),T(pad),T(_np.array([float(_np.median([r['precursor_mz'] for r in rows]))],_np.float32)),
          T(_np.array([ref.ADDUCT_IX.get(m0['adduct'],ref.ADDUCT_IX['<unk>'])])),
          T(_np.array([ref.instr_family(m0['instrument_type'])])),
          T(_np.array([25.0],_np.float32)),T(_np.array([mode],_np.float32)))
    net_merged.eval()
    return net_merged(*args).float().cpu().numpy()[0]
rr = []
for tikey,members in G.items():
    z = merged_z(members)
    if z is None:
        rr.append(None); continue
    qm = neutral_mass(np.array([members[0]['precursor_mz']]), [members[0]['adduct']])[0]
    w = window(qm)
    F = np.unpackbits(pool_fp[w], axis=1)[:, :6930].astype(np.float32)
    order = np.argsort(-(F @ z))
    ti = key_to_idx.get(tikey)
    pos = np.where(w[order]==ti)[0]
    rr.append(int(pos[0])+1 if len(pos) else None)
got=[x for x in rr if x]
print('merged mrr=%.4f top1=%.4f ranked=%d/250' % (sum(1/x for x in got)/len(rr), sum(1 for x in got if x==1)/len(rr), len(got)))
json.dump({'ranks':rr}, open('/content/004-output2/merged_np250.json','w'))


## forward 试点（有界：无已核实权重 URL 则按许可规则跳过并记录）
只用规则允许的版本；NIST 训练权重排除。11 个 R2 硬例 × top-25 的重排留待权重落地。

In [ ]:
import json
ALLOWLIST = {}
reason = 'no verified GLACIER/MassSpecGym weight file URL on record; NIST-trained weights excluded by host ruling 744470'
print('SKIP:', reason)
json.dump({'status':'skipped','reason':reason,'allowlist':ALLOWLIST,
           'needs':['verified weight file URL','ms-pred compatible loader']},
          open('/content/004-output2/forward_pilot.json','w'), indent=1)


## 收尾：下载 `/content/004-output2/` 全部回本地 `artifacts/004-research/`：
`r1_cosine_np1184.json`、`r4_ranker.json`、`r4_np1184.json`、`unfreeze_2000.pt`、
`unfreeze_log.json`、`merged_np250.json`、`forward_pilot.json`。把终端汇总行（cosine/R4/unfreeze/merged）贴回 review。